# Boreal resilience pipeline — Colab full run

Updated for your current datasets:

- **MOD13C2 NDVI** monthly HDF folder
- **VODCA CXKu** pre-aggregated monthly NetCDF files in `monthly_images_VODCA_CXKu/`
- **MOD17 GPP** preprocessed monthly 0.25° NetCDF file `gpp_monthly_025deg_2000_2021.nc`

This notebook is intended for the **full benchmarking run** in Colab.


## Colab setup

Run the next two cells first.  
They install the geospatial packages needed for raw MODIS HDF files and mount Google Drive.


In [1]:
# Colab package install
!apt-get -qq update
!apt-get -qq install -y gdal-bin python3-gdal libgdal-dev > /dev/null
!pip -q install pyhdf xarray netCDF4 dask[complete] rasterio rioxarray pyproj bottleneck scipy statsmodels


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.3/780.3 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 114.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.0 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import glob
import math
import os
import re
import shutil
import tempfile
from collections import defaultdict

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import statsmodels.api as sm

import rasterio
import rioxarray as rxr
from pyhdf.SD import SD, SDC
from pyproj import Transformer
from scipy.stats import spearmanr, pearsonr, linregress
from scipy.optimize import curve_fit
from scipy.signal import savgol_filter, argrelmax
from statsmodels.tsa.seasonal import STL
from osgeo import gdal

xr.set_options(keep_attrs=True)


## 0. Configuration
Set paths and analysis settings.

In [3]:
print("STEP 0 - Configuration (Colab full run, STL + Taylor transitions + GLSAR)")

# ---------- General ----------
SEED = 42
np.random.seed(SEED)

# Edit this to the folder where you keep the project inside Google Drive.
PROJECT_ROOT = Path("/content/drive/MyDrive/Pythonnb")
DATA_DIR = PROJECT_ROOT / "data"
CACHE_DIR = PROJECT_ROOT / "cache_colab"
OUTPUT_DIR = PROJECT_ROOT / "outputs_resilience_stl_taylor_glsar_full_colab"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------- Full-run controls ----------
TEST_RUN = False
MODIS_NDVI_MAX_FILES = None
TEST_MAX_TIME_STEPS = None
TEST_NOTE = "Full notebook: STL preprocessing + Taylor-style transition detection/recovery + GLSAR metric"

# ---------- Region ----------
LAT_MIN, LAT_MAX = 45.0, 75.0
LON_MIN, LON_MAX = -170.0, -50.0

# ---------- Common output grid ----------
COMMON_RES_DEG = 0.25

# ---------- Boreal mask ----------
BOREAL_MASK_FILE = PROJECT_ROOT / "MCD12C1_boreal_mask.nc"   # optional
USE_BOREAL_MASK = BOREAL_MASK_FILE.exists()

# ---------- Sampling ----------
N_LAT_STRATA = 4
N_LON_STRATA = 5
PIXELS_PER_STRATUM = 10   # target ~200 per dataset

# ---------- STL preprocessing ----------
STL_PERIOD = 12
STL_SEASONAL = 13
STL_TREND = 25
STL_ROBUST = True

# ---------- Taylor-style disturbance / recovery ----------
TRANSITION_WIN_SIZE = 24          # months
TRANSITION_PCT = 99               # percentile threshold on smoothed derivative
TRANSITION_WIDTH_ROLL = 18        # as in Taylor-style script
MIN_EVENT_LENGTH = 3              # minimum candidate width (months)
MIN_SEPARATION = 12               # months between candidate transitions
RECOVERY_YEARS = 3
RECOVERY_MONTHS = 12 * RECOVERY_YEARS
LOCAL_MIN_SEARCH_MONTHS = 8

# ---------- Data quality ----------
MIN_POINTS = 24
MIN_VARIANCE = 1e-10
MIN_VALID_FRACTION = 0.70
MIN_VALID_FRAC = MIN_VALID_FRACTION

# ---------- Event-level theoretical metrics ----------
WINDOW_SIZE = 60
WINDOW_STEP = 1
EVENT_WINDOW_MODE = "pre"   # "pre" or "center"

# ---------- Dataset paths ----------
MODIS_NDVI_FOLDER = DATA_DIR / "MOD13C2_061-20260303_192221"
VODCA_PATH = DATA_DIR / "monthly_images_VODCA_CXKu"
MODIS_GPP_PATH = DATA_DIR / "gpp_monthly_025deg_2000_2021.nc"

assert PROJECT_ROOT.exists(), f"PROJECT_ROOT not found: {PROJECT_ROOT}"
assert DATA_DIR.exists(), f"Data folder not found: {DATA_DIR}"
assert MODIS_NDVI_FOLDER.exists(), f"MODIS NDVI folder not found: {MODIS_NDVI_FOLDER}"
assert VODCA_PATH.exists(), f"VODCA monthly folder not found: {VODCA_PATH}"
assert MODIS_GPP_PATH.exists(), f"MODIS GPP monthly file not found: {MODIS_GPP_PATH}"

print("Using PROJECT_ROOT:", PROJECT_ROOT)
print("Using DATA_DIR:", DATA_DIR)
print("Using CACHE_DIR:", CACHE_DIR)
print("Using OUTPUT_DIR:", OUTPUT_DIR)
print("Region:", (LAT_MIN, LAT_MAX, LON_MIN, LON_MAX))
print("Use boreal mask:", USE_BOREAL_MASK)
print("MODIS NDVI folder:", MODIS_NDVI_FOLDER)
print("VODCA monthly folder:", VODCA_PATH)
print("MODIS GPP monthly file:", MODIS_GPP_PATH)
print("Transition window:", TRANSITION_WIN_SIZE, "months")
print("Recovery fit window:", RECOVERY_YEARS, "years")
print("Theoretical metrics window mode:", EVENT_WINDOW_MODE)
print(TEST_NOTE)


STEP 0 - Configuration (Colab full run)
Using PROJECT_ROOT: /content/drive/MyDrive/Pythonnb
Using DATA_DIR: /content/drive/MyDrive/Pythonnb/data
Using CACHE_DIR: /content/drive/MyDrive/Pythonnb/cache_colab
Using OUTPUT_DIR: /content/drive/MyDrive/Pythonnb/outputs_resilience_gpp_full_colab
Region: (45.0, 75.0, -170.0, -50.0)
Use boreal mask: True
MODIS NDVI folder: /content/drive/MyDrive/Pythonnb/data/MOD13C2_061-20260303_192221
VODCA monthly folder: /content/drive/MyDrive/Pythonnb/data/monthly_images_VODCA_CXKu
MODIS GPP monthly file: /content/drive/MyDrive/Pythonnb/data/gpp_monthly_025deg_2000_2021.nc
Colab full notebook: monthly VODCA folder + preprocessed monthly 0.25° GPP file


## 1. Low-level helpers
Utility functions used throughout the notebook.

In [4]:
def find_lat_lon_dims(da: xr.DataArray):
    """Find the latitude and longitude dimension names in a DataArray."""
    lat_dim = None
    lon_dim = None
    for dim in da.dims:
        d = dim.lower()
        if "lat" in d or d.startswith("y"):
            if lat_dim is None:
                lat_dim = dim
        if "lon" in d or d.startswith("x"):
            if lon_dim is None:
                lon_dim = dim
    if lat_dim is None or lon_dim is None:
        raise ValueError(f"Could not find lat/lon dims in dims={da.dims}")
    return lat_dim, lon_dim


def get_lat_lon_coords(da: xr.DataArray, lat_dim: str, lon_dim: str):
    """
    Return latitude and longitude coordinate arrays.
    Works for:
    - datasets that already have lat/lon coords
    - MODIS CMG arrays where coords are missing and must be built from indices
    """
    if lat_dim in da.coords:
        lat = da[lat_dim].values
    else:
        n = da.sizes[lat_dim]
        lat = np.linspace(LAT_MAX - 0.025, LAT_MIN + 0.025, n)

    if lon_dim in da.coords:
        lon = da[lon_dim].values
    else:
        n = da.sizes[lon_dim]
        lon = np.linspace(LON_MIN + 0.025, LON_MAX - 0.025, n)

    lat = np.asarray(lat)
    lon = np.asarray(lon)

    lat = np.squeeze(lat)
    lon = np.squeeze(lon)

    return lat, lon


def normalize_longitudes(ds: xr.Dataset | xr.DataArray):
    """
    Convert longitudes from 0..360 to -180..180 if needed.
    Keeps data sorted by longitude.
    """
    candidate_names = [c for c in list(ds.coords) + list(ds.dims) if "lon" in c.lower()]
    if not candidate_names:
        return ds

    lon_name = candidate_names[0]
    lon_vals = ds[lon_name].values

    if np.nanmax(lon_vals) > 180:
        new_lon = ((lon_vals + 180) % 360) - 180
        ds = ds.assign_coords({lon_name: new_lon}).sortby(lon_name)

    return ds


def safe_lat_lon_subset(ds_or_da, lat_name="lat", lon_name="lon",
                        lat_min=None, lat_max=None, lon_min=None, lon_max=None):
    """
    Subset a Dataset/DataArray by lat/lon using boolean masks.
    This is safer than slice(...) because it works even if coordinates are
    descending, slightly irregular, or not strictly monotonic.
    """
    out = ds_or_da

    if lat_name in out.coords and lat_min is not None and lat_max is not None:
        lat_vals = np.asarray(out[lat_name].values).squeeze()
        if lat_vals.size > 0:
            lat_lo = min(lat_min, lat_max)
            lat_hi = max(lat_min, lat_max)
            lat_mask = (out[lat_name] >= lat_lo) & (out[lat_name] <= lat_hi)
            out = out.where(lat_mask, drop=True)

    if lon_name in out.coords and lon_min is not None and lon_max is not None:
        lon_vals = np.asarray(out[lon_name].values).squeeze()
        if lon_vals.size > 0:
            lon_lo = min(lon_min, lon_max)
            lon_hi = max(lon_min, lon_max)
            lon_mask = (out[lon_name] >= lon_lo) & (out[lon_name] <= lon_hi)
            out = out.where(lon_mask, drop=True)

    return out


def spatial_subset(ds_or_da):
    """Subset to the Canada-Alaska study box."""
    probe = ds_or_da if isinstance(ds_or_da, xr.DataArray) else list(ds_or_da.data_vars.values())[0]
    lat_dim, lon_dim = find_lat_lon_dims(probe)

    if lat_dim in ds_or_da.coords and lon_dim in ds_or_da.coords:
        ds_or_da = safe_lat_lon_subset(
            ds_or_da,
            lat_name=lat_dim,
            lon_name=lon_dim,
            lat_min=LAT_MIN,
            lat_max=LAT_MAX,
            lon_min=LON_MIN,
            lon_max=LON_MAX,
        )

    return ds_or_da


def stl_deseason_detrend(da: xr.DataArray):
    """
    Deseason and detrend a monthly time series with STL.
    Returns residual, trend, seasonal.
    Residual is the working series for event detection and theoretical metrics.
    """
    da = da.dropna("time")
    if da.sizes.get("time", 0) < max(MIN_POINTS, 2 * STL_PERIOD):
        return None, None, None

    values = da.values.astype(float)
    if np.sum(np.isfinite(values)) < max(MIN_POINTS, 2 * STL_PERIOD):
        return None, None, None

    try:
        stl = STL(
            values,
            period=STL_PERIOD,
            seasonal=STL_SEASONAL,
            trend=STL_TREND,
            robust=STL_ROBUST,
        )
        res = stl.fit()
    except Exception:
        return None, None, None

    residual = xr.DataArray(
        res.resid.astype(np.float32),
        coords={"time": da.time.values},
        dims=["time"],
        name="residual"
    )
    trend = xr.DataArray(
        res.trend.astype(np.float32),
        coords={"time": da.time.values},
        dims=["time"],
        name="trend"
    )
    seasonal = xr.DataArray(
        res.seasonal.astype(np.float32),
        coords={"time": da.time.values},
        dims=["time"],
        name="seasonal"
    )
    return residual, trend, seasonal


def valid_series_fraction(da: xr.DataArray) -> float:
    if "time" not in da.dims:
        return 0.0
    return float(da.count("time") / da.sizes["time"])


def lag1_autocorrelation(values: np.ndarray) -> float:
    values = np.asarray(values, dtype=float)
    finite = np.isfinite(values)
    values = values[finite]
    if len(values) < 3:
        return np.nan
    x1 = values[:-1]
    x2 = values[1:]
    if np.std(x1) == 0 or np.std(x2) == 0:
        return np.nan
    return float(np.corrcoef(x1, x2)[0, 1])


def ar1_phi(values: np.ndarray) -> float:
    values = np.asarray(values, dtype=float)
    finite = np.isfinite(values)
    values = values[finite]
    if len(values) < 3:
        return np.nan
    x1 = values[:-1]
    x2 = values[1:]
    if np.std(x1) == 0:
        return np.nan
    phi = np.polyfit(x1, x2, 1)[0]
    return float(phi)


def ar1_lambda(values: np.ndarray) -> float:
    """
    Convert AR(1) persistence into a recovery-rate-like quantity:
    lambda = -ln(phi), valid for 0 < phi < 1
    """
    phi = ar1_phi(values)
    if np.isnan(phi):
        return np.nan
    if phi <= 0 or phi >= 1:
        return np.nan
    return float(-np.log(phi))


def glsar_a(values: np.ndarray) -> float:
    """
    GLSAR-based local stability coefficient inspired by run_fit_a_ar1 in EWS_functions.py.
    """
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) < 8:
        return np.nan
    if np.nanstd(x) < MIN_VARIANCE:
        return np.nan

    xw = x - np.nanmean(x)
    try:
        p0, p1 = np.polyfit(np.arange(len(xw)), xw, 1)
        xw = xw - p0 * np.arange(len(xw)) - p1

        dxw = xw[1:] - xw[:-1]
        exog = sm.add_constant(xw[:-1])
        model = sm.GLSAR(dxw, exog, rho=1)
        results = model.iterative_fit(maxiter=10)
        return float(results.params[1])
    except Exception:
        return np.nan


def safe_spearman(x, y):
    df = pd.DataFrame({"x": x, "y": y}).dropna()
    if len(df) < 3:
        return np.nan, np.nan, len(df)
    r, p = spearmanr(df["x"], df["y"])
    return float(r), float(p), len(df)


def safe_pearson(x, y):
    df = pd.DataFrame({"x": x, "y": y}).dropna()
    if len(df) < 3:
        return np.nan, np.nan, len(df)
    r, p = pearsonr(df["x"], df["y"])
    return float(r), float(p), len(df)


## 2. Data loading functions
Defines the MODIS QA and file-loading helpers.

In [5]:
print("\nSTEP 1 - Data loading + monthly preprocessing")

import datetime as dt

def parse_modis_date_from_filename(filename: str) -> pd.Timestamp:
    m = re.search(r"\.A(\d{4})(\d{3})\.", Path(filename).name)
    if not m:
        raise ValueError(f"Could not parse MODIS date from filename: {filename}")
    year = int(m.group(1))
    doy = int(m.group(2))
    return pd.to_datetime(f"{year}-{doy}", format="%Y-%j")

def find_first_matching_sds(sd_obj, keywords):
    names = list(sd_obj.datasets().keys())
    for name in names:
        lname = name.lower()
        if all(k.lower() in lname for k in keywords):
            return name
    raise KeyError(f"No SDS matching {keywords}. Available SDS names: {names[:20]}")

def open_mod13c2_ndvi_hdf(filepath: str) -> xr.DataArray:
    """
    Read MOD13C2 monthly NDVI from HDF using pyhdf, apply QA, and return a lat/lon DataArray.
    """
    hdf = SD(filepath, SDC.READ)

    ndvi_name = find_first_matching_sds(hdf, ["monthly", "ndvi"])
    qa_name = find_first_matching_sds(hdf, ["monthly", "vi quality"])
    rel_name = find_first_matching_sds(hdf, ["monthly", "pixel reliability"])

    ndvi_raw = hdf.select(ndvi_name)[:].astype(np.int32)
    vi_quality = hdf.select(qa_name)[:].astype(np.uint16)
    pixel_reliability = hdf.select(rel_name)[:].astype(np.int16)

    valid_range_mask = (ndvi_raw >= -2000) & (ndvi_raw <= 10000)
    reliability_mask = (pixel_reliability == 0) | (pixel_reliability == 1)
    modland_qa = vi_quality & 0b11
    modland_mask = (modland_qa == 0) | (modland_qa == 1)
    usefulness = (vi_quality >> 2) & 0b1111
    usefulness_mask = usefulness <= 2

    qa_mask = valid_range_mask & reliability_mask & modland_mask & usefulness_mask
    ndvi = np.where(qa_mask, ndvi_raw.astype(np.float32) * 0.0001, np.nan)

    nlat, nlon = ndvi.shape
    lat = 90 - 0.05 * (np.arange(nlat) + 0.5)
    lon = -180 + 0.05 * (np.arange(nlon) + 0.5)

    da = xr.DataArray(
        ndvi,
        dims=("lat", "lon"),
        coords={"lat": lat.astype(np.float32), "lon": lon.astype(np.float32)},
        name="NDVI"
    )

    da = normalize_longitudes(da)
    da = safe_lat_lon_subset(da, "lat", "lon", LAT_MIN, LAT_MAX, LON_MIN, LON_MAX)

    ts = parse_modis_date_from_filename(filepath)
    da = da.expand_dims(time=[pd.Timestamp(ts.year, ts.month, 1)])
    return da

def standardize_time_dimension(ds: xr.Dataset) -> xr.Dataset:
    if "time" in ds.dims or "time" in ds.coords:
        return ds
    candidate_names = []
    for name in list(ds.dims) + list(ds.coords):
        lname = name.lower()
        if "time" in lname or "date" in lname or lname in ["month", "t"]:
            candidate_names.append(name)
    if candidate_names:
        ds = ds.rename({candidate_names[0]: "time"})
    return ds

def pick_preferred_data_var(ds: xr.Dataset, preferred_names: list[str], dataset_name: str) -> str:
    lower_map = {name.lower(): name for name in ds.data_vars}
    for cand in preferred_names:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    if len(ds.data_vars) == 0:
        raise ValueError(f"{dataset_name}: no data variables found")
    chosen = list(ds.data_vars)[0]
    print(f"{dataset_name}: preferred variable not found; using first variable: {chosen}")
    return chosen

def to_month_start_datetimeindex(time_values) -> pd.DatetimeIndex:
    out = []
    for t in list(time_values):
        if isinstance(t, pd.Timestamp):
            ts = t
        elif isinstance(t, np.datetime64):
            ts = pd.Timestamp(t)
        elif isinstance(t, dt.datetime):
            ts = pd.Timestamp(t)
        elif hasattr(t, "year") and hasattr(t, "month"):
            ts = pd.Timestamp(int(t.year), int(t.month), 1)
        else:
            ts = pd.Timestamp(str(t))
        out.append(pd.Timestamp(ts.year, ts.month, 1))
    return pd.DatetimeIndex(out)

def standardize_monthly_time_coord(da: xr.DataArray) -> xr.DataArray:
    if "time" not in da.coords:
        raise ValueError("DataArray has no time coordinate.")
    da = da.assign_coords(time=to_month_start_datetimeindex(da["time"].values))
    da = da.sortby("time")
    if da.indexes["time"].duplicated().any():
        da = da.groupby("time").mean()
    return da

def load_vodca_monthly(vodca_path: Path) -> xr.DataArray:
    cache_path = CACHE_DIR / ("vodca_monthly_test.nc" if TEST_RUN else "vodca_monthly_full.nc")
    if cache_path.exists():
        print(f"VODCA: using cached monthly file {cache_path.name}")
        ds = xr.open_dataset(cache_path)
        return ds["VODCA"]

    print(f"Reading VODCA monthly files from: {vodca_path}")
    files = sorted(vodca_path.glob("*.nc"))
    if len(files) == 0:
        raise ValueError(f"No NetCDF files found in {vodca_path}")

    print(f"Found {len(files)} monthly files")
    parts = []
    for i, fp in enumerate(files, start=1):
        if i == 1 or i % 24 == 0 or i == len(files):
            print(f"  VODCA file {i}/{len(files)}: {fp.name}")
        ds_single = xr.open_dataset(fp, engine="netcdf4")
        try:
            try:
                ds_single = xr.decode_cf(ds_single)
            except Exception:
                pass
            ds_single = standardize_time_dimension(ds_single)
            ds_single = normalize_longitudes(ds_single)
            ds_single = spatial_subset(ds_single)

            vod_var = pick_preferred_data_var(
                ds_single,
                ["VODCA_CXKu", "VODCA_CXKU", "VI", "vod", "vodca"],
                "VODCA monthly"
            )
            da_single = ds_single[vod_var].astype(np.float32)
            rename_dict = {}
            for dim in da_single.dims:
                if dim.lower() in ["latitude", "y"]:
                    rename_dict[dim] = "lat"
                elif dim.lower() in ["longitude", "x"]:
                    rename_dict[dim] = "lon"
            if rename_dict:
                da_single = da_single.rename(rename_dict)
            da_single = da_single.load()
            parts.append(da_single)
        finally:
            ds_single.close()

    da = xr.concat(parts, dim="time")
    da = standardize_monthly_time_coord(da)

    if TEST_MAX_TIME_STEPS is not None and da.sizes.get("time", 0) > TEST_MAX_TIME_STEPS:
        da = da.isel(time=slice(0, TEST_MAX_TIME_STEPS))

    da.name = "VODCA"
    da.to_dataset(name="VODCA").to_netcdf(cache_path)
    print(f"VODCA: wrote cache {cache_path.name}")
    return da

def load_modis_ndvi_monthly() -> xr.DataArray:
    cache_path = CACHE_DIR / ("modis_ndvi_monthly_test.nc" if TEST_RUN else "modis_ndvi_monthly_full.nc")
    if cache_path.exists():
        print(f"MODIS NDVI: using cached monthly file {cache_path.name}")
        ds = xr.open_dataset(cache_path)
        return ds["NDVI"]

    hdf_files = sorted(glob.glob(str(MODIS_NDVI_FOLDER / "*.hdf")))
    if len(hdf_files) == 0:
        raise FileNotFoundError(f"No MOD13C2 HDF files found in {MODIS_NDVI_FOLDER}")

    if MODIS_NDVI_MAX_FILES is not None:
        hdf_files = hdf_files[:MODIS_NDVI_MAX_FILES]

    print(f"MODIS NDVI: reading {len(hdf_files)} monthly HDF files")
    parts = []
    for i, fp in enumerate(hdf_files, start=1):
        if i == 1 or i % 12 == 0 or i == len(hdf_files):
            print(f"  MODIS NDVI file {i}/{len(hdf_files)}: {Path(fp).name}")
        parts.append(open_mod13c2_ndvi_hdf(fp))

    da = xr.concat(parts, dim="time").sortby("time")
    da = standardize_monthly_time_coord(da)
    da.name = "NDVI"
    da.to_dataset(name="NDVI").to_netcdf(cache_path)
    print(f"MODIS NDVI: wrote cache {cache_path.name}")
    return da

def load_mod17_gpp_monthly_preprocessed(gpp_path: Path) -> xr.DataArray:
    cache_path = CACHE_DIR / ("gpp_monthly_preprocessed_test.nc" if TEST_RUN else "gpp_monthly_preprocessed_full.nc")
    if cache_path.exists():
        print(f"MOD17 GPP: using cached monthly file {cache_path.name}")
        ds = xr.open_dataset(cache_path)
        return ds["GPP"]

    print(f"Reading preprocessed GPP file from: {gpp_path}")
    ds = xr.open_dataset(gpp_path, engine="netcdf4")
    try:
        possible_vars = list(ds.data_vars)
        preferred_names = ["GPP", "gpp", "Gpp_500m"]
        gpp_var = None
        for name in preferred_names:
            for v in possible_vars:
                if v == name or name.lower() in v.lower():
                    gpp_var = v
                    break
            if gpp_var is not None:
                break
        if gpp_var is None:
            gpp_var = possible_vars[0]

        da = ds[gpp_var].astype(np.float32)

        rename_dict = {}
        for dim in da.dims:
            if dim.lower() in ["latitude", "y"]:
                rename_dict[dim] = "lat"
            elif dim.lower() in ["longitude", "x"]:
                rename_dict[dim] = "lon"
        if rename_dict:
            da = da.rename(rename_dict)

        da = normalize_longitudes(da)
        da = safe_lat_lon_subset(da, "lat", "lon", LAT_MIN, LAT_MAX, LON_MIN, LON_MAX)
        da = standardize_monthly_time_coord(da)

        if TEST_MAX_TIME_STEPS is not None and da.sizes.get("time", 0) > TEST_MAX_TIME_STEPS:
            da = da.isel(time=slice(0, TEST_MAX_TIME_STEPS))

        da.name = "GPP"
        da.to_dataset(name="GPP").to_netcdf(cache_path)
        print(f"MOD17 GPP: wrote cache {cache_path.name}")
        return da
    finally:
        ds.close()



STEP 1 - Data loading + monthly preprocessing


## 2b. Load datasets (full run)
Loads the three datasets, aligns them to the VODCA grid, and intersects the full monthly time range.


In [6]:
import pandas as pd
import xarray as xr

print("Loading VODCA monthly from pre-aggregated monthly files...")
vodca_monthly = load_vodca_monthly(VODCA_PATH)
print("VODCA monthly shape:", dict(vodca_monthly.sizes))
print("VODCA time range:", str(vodca_monthly.time.values[0]), "to", str(vodca_monthly.time.values[-1]))

target_lat = vodca_monthly["lat"]
target_lon = vodca_monthly["lon"]

def align_to_target_grid(da: xr.DataArray, target_lat: xr.DataArray, target_lon: xr.DataArray) -> xr.DataArray:
    lat_dim, lon_dim = find_lat_lon_dims(da)
    da = normalize_longitudes(da)
    if lat_dim != "lat" or lon_dim != "lon":
        da = da.rename({lat_dim: "lat", lon_dim: "lon"})
    da = da.sortby("lat").sortby("lon")
    return da.interp(lat=target_lat, lon=target_lon, method="nearest")

print("Loading MODIS NDVI monthly...")
modis_ndvi = load_modis_ndvi_monthly()
modis_ndvi = align_to_target_grid(modis_ndvi, target_lat, target_lon)
modis_ndvi = standardize_monthly_time_coord(modis_ndvi)
print("MODIS NDVI aligned shape:", dict(modis_ndvi.sizes))

print("Loading preprocessed MOD17 GPP monthly 0.25° file...")
gpp_monthly = load_mod17_gpp_monthly_preprocessed(MODIS_GPP_PATH)
gpp_monthly = align_to_target_grid(gpp_monthly, target_lat, target_lon)
gpp_monthly = standardize_monthly_time_coord(gpp_monthly)
print("MOD17 GPP aligned shape:", dict(gpp_monthly.sizes))

vodca_monthly = standardize_monthly_time_coord(vodca_monthly)

common_time = modis_ndvi.indexes["time"].intersection(vodca_monthly.indexes["time"]).intersection(gpp_monthly.indexes["time"])
common_time = pd.DatetimeIndex(common_time).sort_values()

print("Common monthly steps across all three datasets:", len(common_time))
print("Common time start/end:", common_time[0], common_time[-1])

modis_ndvi = modis_ndvi.sel(time=common_time)
vodca_monthly = vodca_monthly.sel(time=common_time)
gpp_monthly = gpp_monthly.sel(time=common_time)

ds_vodca = vodca_monthly.to_dataset(name="VODCA")
ds_gpp = gpp_monthly.to_dataset(name="GPP")
vodca_var = "VODCA"
gpp_var = "GPP"

print("Final dataset shapes on common grid:")
print("  MODIS NDVI:", dict(modis_ndvi.sizes))
print("  VODCA:", dict(ds_vodca[vodca_var].sizes))
print("  GPP:", dict(ds_gpp[gpp_var].sizes))


Loading VODCA monthly from pre-aggregated monthly files...
VODCA: using cached monthly file vodca_monthly_full.nc
VODCA monthly shape: {'time': 264, 'lat': 120, 'lon': 480}
VODCA time range: 2000-01-01T00:00:00.000000000 to 2021-12-01T00:00:00.000000000
Loading MODIS NDVI monthly...
MODIS NDVI: reading 287 monthly HDF files
  MODIS NDVI file 1/287: MOD13C2.A2000032.061.2020042083108.hdf
  MODIS NDVI file 12/287: MOD13C2.A2001001.061.2020061230446.hdf
  MODIS NDVI file 24/287: MOD13C2.A2002001.061.2020068122429.hdf
  MODIS NDVI file 36/287: MOD13C2.A2003001.061.2020090105033.hdf
  MODIS NDVI file 48/287: MOD13C2.A2004001.061.2020119112939.hdf
  MODIS NDVI file 60/287: MOD13C2.A2005001.061.2020216163234.hdf
  MODIS NDVI file 72/287: MOD13C2.A2006001.061.2020255033743.hdf
  MODIS NDVI file 84/287: MOD13C2.A2007001.061.2021056025227.hdf
  MODIS NDVI file 96/287: MOD13C2.A2008001.061.2021088223818.hdf
  MODIS NDVI file 108/287: MOD13C2.A2009001.061.2021125215625.hdf
  MODIS NDVI file 120/28

## 3. Boreal mask
Loads and aligns the boreal forest mask.

In [7]:
print("\nSTEP 2 - Loading boreal forest mask")

def load_boreal_mask_for_grid(target_da: xr.DataArray):
    if not USE_BOREAL_MASK:
        print("No boreal mask file supplied or file not found. Using study-area bounding box only.")
        return None

    print("Loading boreal mask from:", BOREAL_MASK_FILE)
    ds_mask = xr.open_dataset(BOREAL_MASK_FILE)
    boreal_mask = ds_mask["boreal_mask"]

    mask_lat_dim, mask_lon_dim = find_lat_lon_dims(boreal_mask)
    if mask_lat_dim != "lat" or mask_lon_dim != "lon":
        boreal_mask = boreal_mask.rename({mask_lat_dim: "lat", mask_lon_dim: "lon"})

    boreal_mask = boreal_mask.sortby("lat").sortby("lon")

    lat_dim, lon_dim = find_lat_lon_dims(target_da)
    target_lat, target_lon = get_lat_lon_coords(target_da, lat_dim, lon_dim)

    target_lat = np.asarray(target_lat, dtype=float).ravel()
    target_lon = np.asarray(target_lon, dtype=float).ravel()
    target_lat = target_lat[~np.isnan(target_lat)]
    target_lon = target_lon[~np.isnan(target_lon)]

    if target_lat.size == 0 or target_lon.size == 0:
        raise ValueError("Target grid has empty lat/lon coordinates during boreal-mask alignment.")

    boreal_mask_aligned = boreal_mask.interp(
        lat=target_lat,
        lon=target_lon,
        method="nearest"
    )

    rename_dict = {}
    if "lat" in boreal_mask_aligned.dims and lat_dim not in boreal_mask_aligned.dims:
        rename_dict["lat"] = lat_dim
    if "lon" in boreal_mask_aligned.dims and lon_dim not in boreal_mask_aligned.dims:
        rename_dict["lon"] = lon_dim
    if rename_dict:
        boreal_mask_aligned = boreal_mask_aligned.rename(rename_dict)

    boreal_mask_aligned = boreal_mask_aligned.astype(bool)
    print("Boreal mask successfully aligned to target grid.")
    return boreal_mask_aligned


STEP 2 - Loading boreal forest mask


## 4. Stratified sampling functions
Build candidate pixels and sample them by stratum.

In [8]:
print("\nSTEP 3 - Stratified pixel sampling")

def build_candidate_pixel_table(da: xr.DataArray, dataset_name: str):
    """
    Fast vectorized candidate-pixel builder.
    A pixel is a candidate if it has enough valid time points,
    enough valid fraction, enough variance, and (optionally) falls
    inside the boreal mask.
    """
    lat_dim, lon_dim = find_lat_lon_dims(da)

    if da.sizes.get(lat_dim, 0) == 0:
        print(f"{dataset_name}: latitude dimension is empty before sampling. Returning an empty candidate table.")
        return pd.DataFrame(columns=["dataset", "pixel_id", "lat_idx", "lon_idx", "lat", "lon"])
    if da.sizes.get(lon_dim, 0) == 0:
        print(f"{dataset_name}: longitude dimension is empty before sampling. Returning an empty candidate table.")
        return pd.DataFrame(columns=["dataset", "pixel_id", "lat_idx", "lon_idx", "lat", "lon"])

    # Force a consistent dimension order before building 2D masks.
    da = da.transpose("time", lat_dim, lon_dim)
    print(f"{dataset_name}: dims after transpose = {da.dims}")
    print(f"{dataset_name}: shape after transpose = {da.shape}")

    lat_values, lon_values = get_lat_lon_coords(da, lat_dim, lon_dim)
    if np.asarray(lat_values).size == 0 or np.asarray(lon_values).size == 0:
        print(f"{dataset_name}: coordinate arrays are empty after extraction. Returning an empty candidate table.")
        return pd.DataFrame(columns=["dataset", "pixel_id", "lat_idx", "lon_idx", "lat", "lon"])

    print(f"{dataset_name}: loading boreal mask...")
    boreal_mask = load_boreal_mask_for_grid(da)

    print(f"{dataset_name}: computing valid-count / valid-fraction / variance masks...")
    valid_count = da.count("time")
    valid_fraction = valid_count / da.sizes["time"]
    variance = da.var("time", skipna=True)

    candidate_mask = (
        (valid_count >= MIN_POINTS)
        & (valid_fraction >= MIN_VALID_FRACTION)
        & (variance >= MIN_VARIANCE)
    )

    if boreal_mask is not None:
        candidate_mask = candidate_mask & boreal_mask.astype(bool)

    print(f"{dataset_name}: materializing candidate mask into memory...")
    if hasattr(candidate_mask.data, "compute"):
        candidate_mask = candidate_mask.compute()
    else:
        candidate_mask = candidate_mask.load()

    candidate_values = np.asarray(candidate_mask.values, dtype=bool)
    if candidate_values.ndim != 2:
        raise ValueError(f"{dataset_name}: candidate mask is not 2D. Shape={candidate_values.shape}")

    ij = np.argwhere(candidate_values)
    print(f"{dataset_name}: candidate pixels found = {len(ij)}")

    if len(ij) == 0:
        return pd.DataFrame(columns=["dataset", "pixel_id", "lat_idx", "lon_idx", "lat", "lon"])

    rows = pd.DataFrame({
        "dataset": dataset_name,
        "pixel_id": np.arange(len(ij), dtype=int),
        "lat_idx": ij[:, 0].astype(int),
        "lon_idx": ij[:, 1].astype(int),
    })
    rows["lat"] = lat_values[rows["lat_idx"].to_numpy()]
    rows["lon"] = lon_values[rows["lon_idx"].to_numpy()]
    return rows


def add_spatial_strata(candidate_df: pd.DataFrame) -> pd.DataFrame:
    """Assign each candidate pixel to a latitude and longitude stratum."""
    if len(candidate_df) == 0:
        df = candidate_df.copy()
        for col in ["lat_stratum", "lon_stratum", "stratum"]:
            if col not in df.columns:
                df[col] = pd.Series(dtype="object" if col == "stratum" else "int64")
        return df

    df = candidate_df.copy()
    lat_bins = np.linspace(df["lat"].min(), df["lat"].max(), N_LAT_STRATA + 1)
    lon_bins = np.linspace(df["lon"].min(), df["lon"].max(), N_LON_STRATA + 1)

    df["lat_stratum"] = pd.cut(df["lat"], bins=lat_bins, labels=False, include_lowest=True)
    df["lon_stratum"] = pd.cut(df["lon"], bins=lon_bins, labels=False, include_lowest=True)

    df["lat_stratum"] = df["lat_stratum"].fillna(0).astype(int)
    df["lon_stratum"] = df["lon_stratum"].fillna(0).astype(int)
    df["stratum"] = df["lat_stratum"].astype(str) + "_" + df["lon_stratum"].astype(str)
    return df


def stratified_sample_pixels(candidate_df: pd.DataFrame) -> pd.DataFrame:
    if len(candidate_df) == 0:
        return candidate_df.copy()

    sampled_parts = []
    grouped = candidate_df.groupby("stratum", sort=True)

    for k, (stratum, group) in enumerate(grouped, start=1):
        n = min(PIXELS_PER_STRATUM, len(group))
        sampled = group.sample(n=n, random_state=SEED)
        sampled_parts.append(sampled)
        if k == 1 or k % 10 == 0:
            print(f"  sampled strata processed: {k}")

    sampled_df = pd.concat(sampled_parts, ignore_index=True)
    sampled_df = sampled_df.sort_values(["lat_stratum", "lon_stratum", "lat", "lon"]).reset_index(drop=True)
    sampled_df["sampled_pixel_id"] = np.arange(len(sampled_df))
    return sampled_df


def sample_dataset_pixels(da: xr.DataArray, dataset_name: str) -> pd.DataFrame:
    print(f"{dataset_name}: building candidate pixel table...")
    candidates = build_candidate_pixel_table(da, dataset_name)
    print(f"{dataset_name}: assigning strata...")
    candidates = add_spatial_strata(candidates)
    print(f"{dataset_name}: stratified sampling...")
    sampled = stratified_sample_pixels(candidates)
    print(f"{dataset_name}: candidates={len(candidates)}, sampled={len(sampled)}")
    return sampled



STEP 3 - Stratified pixel sampling


## 4b. Run stratified sampling (full run)
Uses the full time range and the full dataset configuration defined above.


In [9]:
print("Loaded dataset summary:")
print("  MODIS:", dict(modis_ndvi.sizes))
print("  VODCA:", dict(ds_vodca[vodca_var].sizes))
print("  GPP:", dict(ds_gpp[gpp_var].sizes))


Loaded dataset summary:
  MODIS: {'time': 263, 'lat': 120, 'lon': 480}
  VODCA: {'time': 263, 'lat': 120, 'lon': 480}
  GPP: {'time': 263, 'lat': 120, 'lon': 480}


In [10]:
print("Sampling MODIS pixels...")
modis_sampled_pixels = sample_dataset_pixels(modis_ndvi, "MODIS")
print("Saving MODIS sampled pixels...")
modis_sampled_pixels.to_csv(OUTPUT_DIR / "modis_sampled_pixels.csv", index=False)

print("Sampling VODCA pixels...")
vodca_sampled_pixels = sample_dataset_pixels(ds_vodca[vodca_var], "VODCA")
print(f"VODCA sampled rows: {len(vodca_sampled_pixels)}")
print("Saving VODCA sampled pixels...")
vodca_sampled_pixels.to_csv(OUTPUT_DIR / "vodca_sampled_pixels.csv", index=False)

print("Sampling GPP pixels...")
gpp_sampled_pixels = sample_dataset_pixels(ds_gpp[gpp_var], "GPP")
print(f"GPP sampled rows: {len(gpp_sampled_pixels)}")
print("Saving GPP sampled pixels...")
gpp_sampled_pixels.to_csv(OUTPUT_DIR / "gpp_sampled_pixels.csv", index=False)

print("Saved sampled pixel tables")


Sampling MODIS pixels...
MODIS: building candidate pixel table...
MODIS: dims after transpose = ('time', 'lat', 'lon')
MODIS: shape after transpose = (263, 120, 480)
MODIS: loading boreal mask...
Loading boreal mask from: /content/drive/MyDrive/Pythonnb/MCD12C1_boreal_mask.nc
Boreal mask successfully aligned to target grid.
MODIS: computing valid-count / valid-fraction / variance masks...
MODIS: materializing candidate mask into memory...
MODIS: candidate pixels found = 11
MODIS: assigning strata...
MODIS: stratified sampling...
  sampled strata processed: 1
MODIS: candidates=11, sampled=11
Saving MODIS sampled pixels...
Sampling VODCA pixels...
VODCA: building candidate pixel table...
VODCA: dims after transpose = ('time', 'lat', 'lon')
VODCA: shape after transpose = (263, 120, 480)
VODCA: loading boreal mask...
Loading boreal mask from: /content/drive/MyDrive/Pythonnb/MCD12C1_boreal_mask.nc
Boreal mask successfully aligned to target grid.
VODCA: computing valid-count / valid-fraction

## 5. Disturbance and recovery functions
Build pixel-level anomaly and recovery records.

In [11]:
print("\nSTEP 4 - Disturbance and recovery extraction (STL + Taylor-style)")

TAYLOR_P0 = [-1.0, -0.1]
TAYLOR_BOUNDS = ([-np.inf, -np.inf], [0.0, 0.0])

def exp_fit(x, a, b):
    return a * np.exp(b * x)

def exp_jac(x, a, b):
    return np.array([np.exp(b * x), a * x * np.exp(b * x)]).T

def get_rsq(obs, pred):
    obs = np.asarray(obs, dtype=float)
    pred = np.asarray(pred, dtype=float)
    ss_res = np.nansum((obs - pred) ** 2)
    ss_tot = np.nansum((obs - np.nanmean(obs)) ** 2)
    if ss_tot == 0:
        return np.nan
    return float(1 - ss_res / ss_tot)

def enforce_min_separation_times(event_times: list[int], min_sep: int) -> list[int]:
    if len(event_times) == 0:
        return []
    kept = [int(event_times[0])]
    for t in event_times[1:]:
        if int(t) - kept[-1] >= min_sep:
            kept.append(int(t))
    return kept

def _smooth_with_nan_support(values: np.ndarray, window: int = 7, polyorder: int = 1) -> np.ndarray:
    arr = np.asarray(values, dtype=float).copy()
    finite = np.isfinite(arr)
    if finite.sum() < max(window, polyorder + 2):
        return arr
    idx = np.arange(len(arr))
    arr[~finite] = np.interp(idx[~finite], idx[finite], arr[finite])
    if window % 2 == 0:
        window += 1
    window = min(window, len(arr) if len(arr) % 2 == 1 else len(arr) - 1)
    if window < polyorder + 2 or window < 3:
        return arr
    return savgol_filter(arr, window_length=window, polyorder=polyorder, deriv=0)

def transition_taylor(x, win_size, pct=99):
    """
    Taylor-style transition detector:
    moving-window derivative contrast, Savitzky-Golay smoothing,
    percentile thresholding, local maxima, and width estimate.
    """
    x = np.asarray(x, dtype=float)
    av_derivative = np.empty(x.shape, dtype=float)
    av_derivative.fill(np.nan)
    half_window = int(win_size / 2)
    ln = x.shape[0]

    for i in range(half_window, ln - half_window):
        av_derivative[i] = np.nanmean(x[i - half_window:i]) - np.nanmean(x[i:i + half_window])

    av_derivative = _smooth_with_nan_support(av_derivative, window=7, polyorder=1)

    av_derivative_masked = av_derivative.copy()
    finite = np.isfinite(av_derivative)
    if finite.sum() == 0:
        return np.array([], dtype=int), []
    thr = np.nanpercentile(av_derivative[finite], pct)
    av_derivative_masked[av_derivative < thr] = 0

    transition_times = argrelmax(np.nan_to_num(av_derivative_masked, nan=0.0), order=1)[0]

    av = av_derivative_masked.copy()
    av[av > 0] = 1
    av[np.isnan(av)] = 0
    cs = pd.Series(av).rolling(TRANSITION_WIDTH_ROLL, min_periods=1).sum()

    widths = []
    half_roll = TRANSITION_WIDTH_ROLL // 2
    for t in transition_times:
        lo = max(0, t - half_roll)
        hi = min(len(cs), t + half_roll)
        width = np.nanmax(cs.values[lo:hi])
        widths.append(float(width) if np.isfinite(width) else np.nan)

    return transition_times.astype(int), widths

def fit_transition_taylor(transition_idx: int, detrended_resid: pd.Series, raw: pd.Series, width_months: int | None = None):
    """
    Taylor-style empirical recovery fitting on deseasoned+detrended data.
    Recovery fit window shortened from 5 years to RECOVERY_YEARS.
    """
    transition_date = pd.Timestamp(detrended_resid.index[transition_idx])
    end_date = transition_date + pd.DateOffset(months=RECOVERY_MONTHS)
    prev_date = transition_date - pd.DateOffset(months=RECOVERY_MONTHS)

    if end_date > pd.Timestamp(detrended_resid.index.max()):
        return None

    fitting = detrended_resid[(detrended_resid.index >= transition_date) & (detrended_resid.index <= end_date)].copy()
    fitting_raw = raw[(raw.index >= transition_date) & (raw.index <= end_date)].copy()

    if len(fitting) < MIN_EVENT_LENGTH:
        return None

    look_ahead = min(LOCAL_MIN_SEARCH_MONTHS, len(fitting))
    try:
        armin = int(np.nanargmin(fitting.values[:look_ahead]))
    except Exception:
        return None

    fitting_min = fitting.iloc[armin:].copy()
    fitting_raw_min = fitting_raw.iloc[armin:].copy()

    if len(fitting_min) < MIN_EVENT_LENGTH:
        return None

    raw_drop = float(fitting_raw.iloc[0] - fitting_raw.iloc[armin])

    centered = fitting_min.values.astype(float) - np.nanmean(fitting_min.values.astype(float))
    finite = np.isfinite(centered)
    if finite.sum() < MIN_EVENT_LENGTH:
        return None

    trange = np.arange(finite.sum(), dtype=float)
    y = centered[finite]

    try:
        popt, _ = curve_fit(
            exp_fit,
            trange,
            y,
            p0=TAYLOR_P0,
            jac=exp_jac,
            bounds=TAYLOR_BOUNDS,
            maxfev=10000
        )
        pred = exp_fit(trange, *popt)
        rsq = get_rsq(y, pred)
        fit_b = float(popt[1])   # <= 0
    except Exception:
        return None

    lambda_emp = -fit_b
    if not np.isfinite(lambda_emp) or lambda_emp <= 0:
        return None

    prevser = raw[(raw.index < transition_date) & (raw.index >= prev_date)].copy()
    returned_to_baseline = np.nan
    if len(prevser) >= 3 and len(fitting_raw_min) >= 1:
        prev_mean = float(np.nanmean(prevser.values))
        prev_std = float(np.nanstd(prevser.values))
        returned_to_baseline = bool(np.nanmean(fitting_raw_min.tail(min(3, len(fitting_raw_min))).values) >= (prev_mean - prev_std))

    duration = int(width_months) if width_months is not None and np.isfinite(width_months) else np.nan
    if np.isfinite(duration):
        event_end_idx = min(len(raw.index) - 1, transition_idx + max(int(duration) - 1, 0))
        event_end_time = pd.Timestamp(raw.index[event_end_idx])
    else:
        event_end_time = transition_date

    return {
        "event_start_time": transition_date,
        "event_end_time": event_end_time,
        "disturbance_duration_months": duration,
        "trough_time": pd.Timestamp(fitting.index[armin]),
        "trough_value": float(fitting.iloc[armin]),
        "recovery_end_time": pd.Timestamp(fitting_min.index[-1]),
        "recovery_duration_months": int(len(fitting_min)),
        "returned_to_baseline": returned_to_baseline,
        "lambda_emp": float(lambda_emp),
        "lambda_emp_r2": float(rsq) if np.isfinite(rsq) else np.nan,
        "disturbance_magnitude": float(abs(raw_drop)),
        "transition_width_months": float(width_months) if width_months is not None and np.isfinite(width_months) else np.nan
    }

def build_pixel_record(da: xr.DataArray, row: pd.Series, dataset_name: str) -> dict | None:
    """
    Build one pixel record using STL residuals as the working series and
    Taylor-style transition detection / recovery fitting.
    """
    lat_dim, lon_dim = find_lat_lon_dims(da)
    ts = da.isel({lat_dim: int(row["lat_idx"]), lon_dim: int(row["lon_idx"])}).dropna("time")
    if ts.sizes.get("time", 0) < MIN_POINTS:
        return None

    residual, trend, seasonal = stl_deseason_detrend(ts)
    if residual is None:
        return None

    resid_values = residual.values.astype(float)
    resid_times = pd.to_datetime(residual.time.values)
    finite = np.isfinite(resid_values)
    resid_values = resid_values[finite]
    resid_times = resid_times[finite]

    if len(resid_values) < MIN_POINTS or np.nanvar(resid_values) < MIN_VARIANCE:
        return None

    residual_clean = xr.DataArray(resid_values, coords={"time": resid_times}, dims=["time"], name="residual")
    raw_clean = ts.sel(time=resid_times).astype(np.float32)

    transition_times_raw, widths_raw = transition_taylor(resid_values, TRANSITION_WIN_SIZE, pct=TRANSITION_PCT)
    candidates = []
    for t, w in zip(list(transition_times_raw), list(widths_raw)):
        if np.isfinite(w) and w >= MIN_EVENT_LENGTH:
            candidates.append((int(t), float(w)))
    kept_times = enforce_min_separation_times([t for t, _ in candidates], MIN_SEPARATION)
    kept_lookup = {t: w for t, w in candidates if t in kept_times}

    resid_ser = pd.Series(resid_values, index=pd.DatetimeIndex(resid_times))
    raw_ser = pd.Series(raw_clean.values.astype(float), index=pd.DatetimeIndex(resid_times))

    event_rows = []
    for t in kept_times:
        width = kept_lookup.get(int(t), np.nan)
        if np.isfinite(width) and width < MIN_EVENT_LENGTH:
            continue

        event = fit_transition_taylor(
            transition_idx=int(t),
            detrended_resid=resid_ser,
            raw=raw_ser,
            width_months=width,
        )
        if event is None:
            continue
        event_rows.append(event)

    events_df = pd.DataFrame(event_rows)

    return {
        "dataset": dataset_name,
        "pixel_id": int(row["sampled_pixel_id"]),
        "lat_idx": int(row["lat_idx"]),
        "lon_idx": int(row["lon_idx"]),
        "lat": float(row["lat"]),
        "lon": float(row["lon"]),
        "raw": raw_clean,
        "trend": trend,
        "seasonal": seasonal,
        "anomaly": residual_clean,   # kept for downstream compatibility
        "events_df": events_df,
        "n_events": int(len(events_df))
    }

def build_pixel_records_for_dataset(da: xr.DataArray, sampled_df: pd.DataFrame, dataset_name: str) -> list[dict]:
    records = []
    total = len(sampled_df)
    for idx_row, (_, row) in enumerate(sampled_df.iterrows(), start=1):
        if idx_row == 1 or idx_row % 25 == 0 or idx_row == total:
            print(f"  {dataset_name}: processing sampled pixel {idx_row}/{total}")
        rec = build_pixel_record(da, row, dataset_name)
        if rec is not None:
            records.append(rec)
    print(f"{dataset_name}: valid pixel records={len(records)}")
    return records



STEP 4 - Disturbance and recovery extraction


## 5b. Build pixel records
Reduced test run: still potentially slow, but much lighter than the full notebook.


In [12]:
print("Building MODIS pixel records...")
modis_records = build_pixel_records_for_dataset(modis_ndvi, modis_sampled_pixels, "MODIS")
print("Building VODCA pixel records...")
vodca_records = build_pixel_records_for_dataset(ds_vodca[vodca_var], vodca_sampled_pixels, "VODCA")
print("Building GPP pixel records...")
gpp_records = build_pixel_records_for_dataset(ds_gpp[gpp_var], gpp_sampled_pixels, "GPP")


Building MODIS pixel records...
  MODIS: processing sampled pixel 1/11
  MODIS: processing sampled pixel 11/11
MODIS: valid pixel records=11
Building VODCA pixel records...
  VODCA: processing sampled pixel 1/73
  VODCA: processing sampled pixel 25/73
  VODCA: processing sampled pixel 50/73
  VODCA: processing sampled pixel 73/73
VODCA: valid pixel records=73
Building GPP pixel records...
  GPP: processing sampled pixel 1/130
  GPP: processing sampled pixel 25/130
  GPP: processing sampled pixel 50/130
  GPP: processing sampled pixel 75/130
  GPP: processing sampled pixel 100/130
  GPP: processing sampled pixel 125/130
  GPP: processing sampled pixel 130/130
GPP: valid pixel records=130


## 6. Event-level theoretical metrics
Compute local variance, AC1, and AR-based lambda around each event.

In [13]:
print("\nSTEP 5 - Event-level theoretical metrics (including a_GLSAR)")

def get_event_window(residual_da: xr.DataArray, center_time: pd.Timestamp, window_months: int = 60, mode: str = "pre"):
    if mode == "center":
        start = center_time - pd.DateOffset(months=window_months // 2)
        end = center_time + pd.DateOffset(months=window_months // 2)
    else:
        end = center_time
        start = center_time - pd.DateOffset(months=window_months)
    seg = residual_da.sel(time=slice(start, end)).dropna("time")
    return seg

def compute_local_theoretical_metrics(anomaly_da: xr.DataArray, center_time: pd.Timestamp, window_months: int = 60, mode: str = "pre"):
    """
    Compute theoretical metrics on an STL-residual window.
    Default is a pre-event window, but center-based windows can also be used.
    """
    seg = get_event_window(anomaly_da, center_time=center_time, window_months=window_months, mode=mode)
    npts = int(seg.sizes.get("time", 0))
    if npts < max(24, window_months // 2):
        return {
            "variance": np.nan,
            "ac1": np.nan,
            "lambda_ar1": np.nan,
            "a_glsar": np.nan,
            "window_points": npts
        }

    values = seg.values.astype(float)
    if np.nanvar(values) < MIN_VARIANCE:
        return {
            "variance": np.nan,
            "ac1": np.nan,
            "lambda_ar1": np.nan,
            "a_glsar": np.nan,
            "window_points": npts
        }

    return {
        "variance": float(np.nanvar(values)),
        "ac1": lag1_autocorrelation(values),
        "lambda_ar1": ar1_lambda(values),
        "a_glsar": glsar_a(values),
        "window_points": npts
    }


def build_event_level_table(records: list[dict], dataset_name: str) -> pd.DataFrame:
    expected_cols = [
        "dataset",
        "pixel_id",
        "lat",
        "lon",
        "event_id",
        "event_start_time",
        "event_end_time",
        "trough_time",
        "recovery_end_time",
        "disturbance_duration_months",
        "recovery_duration_months",
        "returned_to_baseline",
        "disturbance_magnitude",
        "transition_width_months",
        "lambda_emp",
        "lambda_emp_r2",
        "variance",
        "ac1",
        "lambda_ar1",
        "a_glsar",
        "window_points",
    ]

    rows = []

    total = len(records)
    for idx_rec, rec in enumerate(records, start=1):
        if idx_rec == 1 or idx_rec % 25 == 0 or idx_rec == total:
            print(f"  {dataset_name}: event metrics for pixel record {idx_rec}/{total}")

        anomaly_da = rec["anomaly"]
        events_df = rec["events_df"]

        if events_df is None or len(events_df) == 0:
            continue

        for event_idx, event in events_df.iterrows():
            local = compute_local_theoretical_metrics(
                anomaly_da=anomaly_da,
                center_time=pd.Timestamp(event["trough_time"]),
                window_months=WINDOW_SIZE,
                mode=EVENT_WINDOW_MODE,
            )

            rows.append({
                "dataset": rec["dataset"],
                "pixel_id": rec["pixel_id"],
                "lat": rec["lat"],
                "lon": rec["lon"],
                "event_id": int(event_idx),
                "event_start_time": event["event_start_time"],
                "event_end_time": event["event_end_time"],
                "trough_time": event["trough_time"],
                "recovery_end_time": event["recovery_end_time"],
                "disturbance_duration_months": event["disturbance_duration_months"],
                "recovery_duration_months": event["recovery_duration_months"],
                "returned_to_baseline": event["returned_to_baseline"],
                "disturbance_magnitude": event["disturbance_magnitude"],
                "transition_width_months": event.get("transition_width_months", np.nan),
                "lambda_emp": event["lambda_emp"],
                "lambda_emp_r2": event["lambda_emp_r2"],
                "variance": local["variance"],
                "ac1": local["ac1"],
                "lambda_ar1": local["lambda_ar1"],
                "a_glsar": local["a_glsar"],
                "window_points": local["window_points"]
            })

    return pd.DataFrame(rows, columns=expected_cols)



STEP 5 - Event-level theoretical metrics


## 6b. Build event-level benchmark table

In [14]:
print("Building event-level table for MODIS...")
events_modis = build_event_level_table(modis_records, "MODIS")
print("Building event-level table for VODCA...")
events_vodca = build_event_level_table(vodca_records, "VODCA")
print("Building event-level table for GPP...")
events_gpp = build_event_level_table(gpp_records, "GPP")

events_all = pd.concat([events_modis, events_vodca, events_gpp], ignore_index=True)
events_all.to_csv(OUTPUT_DIR / "event_level_benchmark_table.csv", index=False)

print("MODIS events:", len(events_modis))
print("VODCA events:", len(events_vodca))
print("GPP events:", len(events_gpp))
print("Total valid events:", len(events_all))
print("Columns:", list(events_all.columns))
print(events_all.head())


Building event-level table for MODIS...
  MODIS: event metrics for pixel record 1/11
  MODIS: event metrics for pixel record 11/11
Building event-level table for VODCA...
  VODCA: event metrics for pixel record 1/73
  VODCA: event metrics for pixel record 25/73
  VODCA: event metrics for pixel record 50/73
  VODCA: event metrics for pixel record 73/73
Building event-level table for GPP...
  GPP: event metrics for pixel record 1/130
  GPP: event metrics for pixel record 25/130
  GPP: event metrics for pixel record 50/130
  GPP: event metrics for pixel record 75/130
  GPP: event metrics for pixel record 100/130
  GPP: event metrics for pixel record 125/130
  GPP: event metrics for pixel record 130/130
MODIS events: 2
VODCA events: 20
GPP events: 61
Total valid events: 83
Columns: ['dataset', 'pixel_id', 'lat', 'lon', 'event_id', 'event_start_time', 'event_end_time', 'trough_time', 'recovery_end_time', 'disturbance_duration_months', 'recovery_duration_months', 'returned_to_baseline', 'dis

## 7. Pixel-level summary table

In [15]:
print("\nSTEP 6 - Pixel-level summary table")

def build_pixel_summary(records: list[dict]) -> pd.DataFrame:
    rows = []
    for rec in records:
        values = rec["anomaly"].values.astype(float)
        rows.append({
            "dataset": rec["dataset"],
            "pixel_id": rec["pixel_id"],
            "lat": rec["lat"],
            "lon": rec["lon"],
            "n_events": rec["n_events"],
            "variance_full_series": float(np.var(values)) if len(values) > 0 else np.nan,
            "ac1_full_series": lag1_autocorrelation(values),
            "lambda_ar1_full_series": ar1_lambda(values)
        })
    return pd.DataFrame(rows)

print("Building pixel summary table...")
pixel_summary = pd.concat([
    build_pixel_summary(modis_records),
    build_pixel_summary(vodca_records),
    build_pixel_summary(gpp_records)
], ignore_index=True)

pixel_summary.to_csv(OUTPUT_DIR / "pixel_summary.csv", index=False)
print("Saved pixel summary table")



STEP 6 - Pixel-level summary table
Building pixel summary table...
Saved pixel summary table


## 8. Benchmarking theoretical metrics against empirical recovery

In [16]:
print("\nSTEP 7 - Proper benchmarking")

def benchmark_metric(df: pd.DataFrame, metric_col: str, metric_name: str) -> dict:
    """
    Benchmark one theoretical metric against lambda_emp.
    """
    tmp = df[["lambda_emp", metric_col]].dropna()
    n = len(tmp)

    if n < 3:
        return {
            "metric": metric_name,
            "n_events": n,
            "spearman_r": np.nan,
            "spearman_p": np.nan,
            "pearson_r": np.nan,
            "pearson_p": np.nan,
            "mae_after_sign_alignment": np.nan,
            "coverage": n / max(len(df), 1)
        }

    sp_r, sp_p, _ = safe_spearman(tmp[metric_col], tmp["lambda_emp"])
    pr_r, pr_p, _ = safe_pearson(tmp[metric_col], tmp["lambda_emp"])

    aligned_metric = tmp[metric_col].copy()
    if metric_name in ["variance", "ac1"]:
        aligned_metric = -aligned_metric

    mae = float(np.mean(np.abs(aligned_metric - tmp["lambda_emp"])))

    return {
        "metric": metric_name,
        "n_events": n,
        "spearman_r": sp_r,
        "spearman_p": sp_p,
        "pearson_r": pr_r,
        "pearson_p": pr_p,
        "mae_after_sign_alignment": mae,
        "coverage": n / max(len(df), 1)
    }


def benchmark_dataset(df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    if "dataset" not in df.columns:
        return pd.DataFrame(columns=[
            "metric", "n_events", "spearman_r", "spearman_p",
            "pearson_r", "pearson_p", "mae_after_sign_alignment",
            "coverage", "dataset"
        ])

    dataset_df = df[df["dataset"] == dataset_name].copy()

    results = []
    for metric_col, metric_name in [
        ("variance", "variance"),
        ("ac1", "ac1"),
        ("lambda_ar1", "lambda_ar1"),
        ("a_glsar", "a_glsar"),
    ]:
        out = benchmark_metric(dataset_df, metric_col, metric_name)
        out["dataset"] = dataset_name
        results.append(out)

    return pd.DataFrame(results)


print("Benchmarking MODIS, VODCA, and GPP...")

if len(events_all) == 0:
    print("No valid events available for benchmarking in this run.")
    benchmark_results = pd.DataFrame(columns=[
        "metric", "n_events", "spearman_r", "spearman_p",
        "pearson_r", "pearson_p", "mae_after_sign_alignment",
        "coverage", "dataset"
    ])
else:
    benchmark_results = pd.concat([
        benchmark_dataset(events_all, "MODIS"),
        benchmark_dataset(events_all, "VODCA"),
        benchmark_dataset(events_all, "GPP"),
    ], ignore_index=True)

benchmark_results.to_csv(OUTPUT_DIR / "benchmark_results.csv", index=False)
print(benchmark_results)



STEP 7 - Proper benchmarking
Benchmarking MODIS, VODCA, and GPP...
       metric  n_events  spearman_r  spearman_p  pearson_r  pearson_p  \
0    variance         2         NaN         NaN        NaN        NaN   
1         ac1         2         NaN         NaN        NaN        NaN   
2  lambda_ar1         2         NaN         NaN        NaN        NaN   
3    variance        20    0.353383    0.126407   0.435871   0.054720   
4         ac1        20    0.087218    0.714638   0.191406   0.418864   
5  lambda_ar1        20   -0.037594    0.874965  -0.136897   0.564939   
6    variance        61   -0.063882    0.624764  -0.041217   0.752464   
7         ac1        61    0.062348    0.633117  -0.053346   0.683040   
8  lambda_ar1        60   -0.110031    0.402643  -0.018375   0.889172   

   mae_after_sign_alignment  coverage dataset  
0                       NaN  1.000000   MODIS  
1                       NaN  1.000000   MODIS  
2                       NaN  1.000000   MODIS  
3        

## 9. Ranking dataset-metric combinations

In [17]:
print("\nSTEP 8 - Ranking dataset-metric combinations")

def minmax_scale(series: pd.Series) -> pd.Series:
    s = series.astype(float).copy()
    finite = np.isfinite(s)
    if finite.sum() == 0:
        return pd.Series(np.nan, index=series.index)
    mn = np.nanmin(s)
    mx = np.nanmax(s)
    if mx == mn:
        out = pd.Series(1.0, index=series.index)
        out[~finite] = np.nan
        return out
    out = (s - mn) / (mx - mn)
    out[~finite] = np.nan
    return out


ranking = benchmark_results.copy()

# Higher absolute Spearman correlation is better
ranking["score_corr"] = np.abs(ranking["spearman_r"])

# Lower MAE is better, so convert to "higher is better"
mae_scaled = minmax_scale(ranking["mae_after_sign_alignment"])
ranking["score_mae"] = 1 - mae_scaled

# coverage already higher is better
ranking["score_cov"] = ranking["coverage"]

# final weighted score
ranking["overall_score"] = (
    0.60 * ranking["score_corr"].fillna(0) +
    0.25 * ranking["score_mae"].fillna(0) +
    0.15 * ranking["score_cov"].fillna(0)
)

ranking = ranking.sort_values("overall_score", ascending=False).reset_index(drop=True)
ranking.to_csv(OUTPUT_DIR / "dataset_metric_ranking.csv", index=False)

print(ranking)

best_row = ranking.iloc[0].to_dict() if len(ranking) > 0 else None
print("\nBest combination:", best_row)
print("Saved ranking table")



STEP 8 - Ranking dataset-metric combinations
       metric  n_events  spearman_r  spearman_p  pearson_r  pearson_p  \
0    variance        20    0.353383    0.126407   0.435871   0.054720   
1  lambda_ar1        60   -0.110031    0.402643  -0.018375   0.889172   
2    variance        61   -0.063882    0.624764  -0.041217   0.752464   
3         ac1        20    0.087218    0.714638   0.191406   0.418864   
4  lambda_ar1        20   -0.037594    0.874965  -0.136897   0.564939   
5         ac1        61    0.062348    0.633117  -0.053346   0.683040   
6    variance         2         NaN         NaN        NaN        NaN   
7         ac1         2         NaN         NaN        NaN        NaN   
8  lambda_ar1         2         NaN         NaN        NaN        NaN   

   mae_after_sign_alignment  coverage dataset  score_corr  score_mae  \
0                  0.384582  1.000000   VODCA    0.353383   1.000000   
1                  0.591375  0.983607     GPP    0.110031   0.704202   
2      

## 10. Sliding-window analysis functions

In [18]:
print("\nSTEP 9 - Sliding window analysis")

def compute_rolling_metrics_for_record(rec: dict) -> pd.DataFrame:
    anomaly_da = rec["anomaly"].dropna("time")
    values = anomaly_da.values.astype(float)
    times = pd.to_datetime(anomaly_da.time.values)

    rows = []
    if len(values) < WINDOW_SIZE:
        return pd.DataFrame(rows)

    for i in range(0, len(values) - WINDOW_SIZE + 1, WINDOW_STEP):
        seg = values[i:i + WINDOW_SIZE]
        seg_times = times[i:i + WINDOW_SIZE]

        if np.nanvar(seg) < MIN_VARIANCE:
            continue

        if EVENT_WINDOW_MODE == "pre":
            center_time = pd.Timestamp(seg_times[-1])
        else:
            center_time = pd.Timestamp(seg_times[len(seg_times) // 2])

        rows.append({
            "dataset": rec["dataset"],
            "pixel_id": rec["pixel_id"],
            "lat": rec["lat"],
            "lon": rec["lon"],
            "time": center_time,
            "variance": float(np.nanvar(seg)),
            "ac1": lag1_autocorrelation(seg),
            "lambda_ar1": ar1_lambda(seg),
            "a_glsar": glsar_a(seg),
        })

    return pd.DataFrame(rows)



STEP 9 - Sliding window analysis


## 10b. Run sliding-window analysis
Full run: uses `WINDOW_SIZE = 60` on the full monthly record.


In [19]:
def compute_rolling_metrics_for_dataset(records: list[dict], dataset_name: str) -> pd.DataFrame:
    parts = []
    total = len(records)
    for idx_rec, rec in enumerate(records, start=1):
        if idx_rec == 1 or idx_rec % 25 == 0 or idx_rec == total:
            print(f"  {dataset_name}: rolling metrics for pixel record {idx_rec}/{total}")
        parts.append(compute_rolling_metrics_for_record(rec))
    if len(parts) == 0:
        return pd.DataFrame()
    return pd.concat(parts, ignore_index=True)

print("Computing rolling metrics for MODIS...")
rolling_modis = compute_rolling_metrics_for_dataset(modis_records, "MODIS")
print("Computing rolling metrics for VODCA...")
rolling_vodca = compute_rolling_metrics_for_dataset(vodca_records, "VODCA")
print("Computing rolling metrics for GPP...")
rolling_gpp = compute_rolling_metrics_for_dataset(gpp_records, "GPP")

rolling_all = pd.concat([
    rolling_modis,
    rolling_vodca,
    rolling_gpp,
], ignore_index=True)

rolling_all.to_csv(OUTPUT_DIR / "rolling_metrics_all.csv", index=False)
print("Rolling rows:", len(rolling_all))


Computing rolling metrics for MODIS...
  MODIS: rolling metrics for pixel record 1/11
  MODIS: rolling metrics for pixel record 11/11
Computing rolling metrics for VODCA...
  VODCA: rolling metrics for pixel record 1/73
  VODCA: rolling metrics for pixel record 25/73
  VODCA: rolling metrics for pixel record 50/73
  VODCA: rolling metrics for pixel record 73/73
Computing rolling metrics for GPP...
  GPP: rolling metrics for pixel record 1/130
  GPP: rolling metrics for pixel record 25/130
  GPP: rolling metrics for pixel record 50/130
  GPP: rolling metrics for pixel record 75/130
  GPP: rolling metrics for pixel record 100/130
  GPP: rolling metrics for pixel record 125/130
  GPP: rolling metrics for pixel record 130/130
Rolling rows: 38852


## 11. Figures and final outputs

In [20]:
print("\nSTEP 10 - Figures")

def save_line_plot(df: pd.DataFrame, value_col: str, title: str, ylabel: str, filename: str):
    plot_df = df.groupby(["dataset", "time"])[value_col].median().reset_index()

    plt.figure(figsize=(12, 6))
    for dataset in plot_df["dataset"].unique():
        tmp = plot_df[plot_df["dataset"] == dataset]
        plt.plot(tmp["time"], tmp[value_col], linewidth=2, label=dataset)

    plt.title(title)
    plt.xlabel("Time")
    plt.ylabel(ylabel)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / filename, dpi=300)
    plt.close()


if len(rolling_all) > 0:
    print("Saving rolling-metric figures...")
    rolling_all["time"] = pd.to_datetime(rolling_all["time"])

    save_line_plot(
        rolling_all,
        value_col="ac1",
        title="Rolling AC1 (median across sampled pixels)",
        ylabel="AC1",
        filename="figure_rolling_ac1.png"
    )

    save_line_plot(
        rolling_all,
        value_col="variance",
        title="Rolling variance (median across sampled pixels)",
        ylabel="Variance",
        filename="figure_rolling_variance.png"
    )

    save_line_plot(
        rolling_all,
        value_col="lambda_ar1",
        title="Rolling AR-based recovery rate (median across sampled pixels)",
        ylabel="AR-based lambda",
        filename="figure_rolling_lambda_ar1.png"
    )

    save_line_plot(
        rolling_all,
        value_col="a_glsar",
        title="Rolling a_GLSAR (median across sampled pixels)",
        ylabel="a_GLSAR",
        filename="figure_rolling_a_glsar.png"
    )

if len(ranking) > 0:
    print("Saving benchmark ranking figure...")
    plt.figure(figsize=(11, 6))
    labels = ranking["dataset"] + " | " + ranking["metric"]
    plt.bar(labels, ranking["overall_score"])
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Overall benchmark score")
    plt.title("Ranking of dataset-metric combinations")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "figure_dataset_metric_ranking.png", dpi=300)
    plt.close()

event_counts = events_all.groupby("dataset").size().reset_index(name="n_events")
event_counts.to_csv(OUTPUT_DIR / "event_counts_by_dataset.csv", index=False)

print("\nSaved outputs in:", OUTPUT_DIR)
print("Pipeline completed successfully.")



STEP 10 - Figures
Saving rolling-metric figures...
Saving benchmark ranking figure...

Saved outputs in: /content/drive/MyDrive/Pythonnb/outputs_resilience_gpp_full_colab
Pipeline completed successfully.
